# ORPO Full Training (PEFT LoRA, Colab)

This notebook is the **refined final training notebook** for your Week 11 Path B run. It avoids the unstable Unsloth LoRA wrapping path you hit earlier and instead uses **explicit PEFT LoRA wrapping** so the final artifact can be saved as a true adapter-only export.


## Run Notes

- Use a Colab GPU runtime such as **T4 GPU**.
- Run the install cell first, then **restart the Colab session** before continuing.
- This notebook trains on the real public ORPO preference file at `training_data/orpo_preferences.jsonl`.
- The goal is a clean adapter-only save with files like `adapter_config.json` and `adapter_model.safetensors`.
- If you see `FastLanguageModel.get_peft_model(...)` anywhere in Colab, stop: that is an older notebook copy, not this final version.


In [ ]:
# Install a PEFT / TRL stack aimed at reliable adapter-only export.
# After this cell finishes, use Runtime -> Restart session, then continue from the next cell.

!pip uninstall -y unsloth unsloth_zoo trl peft transformers bitsandbytes torchao
!pip install --upgrade pip
!pip install --upgrade \
    "transformers>=4.52.0" \
    "peft>=0.17.0" \
    "trl>=1.1.0" \
    "accelerate>=1.7.0" \
    "datasets>=3.6.0" \
    "bitsandbytes>=0.48.0" \
    "torchao>=0.16.0" \
    matplotlib


In [ ]:
import importlib.metadata as metadata
import platform
import sys
import torch

packages = [
    "trl",
    "peft",
    "transformers",
    "torch",
    "datasets",
    "accelerate",
    "bitsandbytes",
    "torchao",
]
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not-installed"

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": versions,
})

!nvidia-smi

print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("gpu: none")

assert torch.cuda.is_available(), (
    "CUDA is not available. In Colab, go to Runtime -> Change runtime type -> T4 GPU, "
    "save, restart the session, and rerun from the top."
)


## Repo Access

Pick one mode:

1. `drive`: use a copy already stored in Google Drive.
2. `clone`: clone the repo directly into Colab.

Set the values below before running the cell.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_MODE = "clone"  # "drive" or "clone"
DRIVE_REPO_PATH = "/content/drive/MyDrive/Sales Eval Bench"
CLONE_URL = "https://github.com/nebiyou27/sales-eval-bench.git"
CLONE_DIR = "/content/Sales Eval Bench"

if REPO_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    repo_root = Path(DRIVE_REPO_PATH)
elif REPO_MODE == "clone":
    repo_root = Path(CLONE_DIR)
    if not repo_root.exists():
        subprocess.run(["git", "clone", CLONE_URL, str(repo_root)], check=True)
else:
    raise ValueError(f"Unsupported REPO_MODE: {REPO_MODE}")

assert repo_root.exists(), f"Repo root not found: {repo_root}"
os.chdir(repo_root)
print(f"repo_root={repo_root}")


## Load And Validate ORPO Training Data

This notebook must train on the real public ORPO corpus and must not use the 5-row smoke fixture.


In [ ]:
import json
from datasets import Dataset

FULL_DATASET_PATH = repo_root / "training_data" / "orpo_preferences.jsonl"
SMOKE_DATASET_PATH = repo_root / "tenacious_bench_v0.1" / "smoke" / "dummy_orpo_preferences.jsonl"

assert FULL_DATASET_PATH.exists(), f"Full ORPO dataset not found: {FULL_DATASET_PATH}"
assert SMOKE_DATASET_PATH.exists(), f"Smoke dataset missing unexpectedly: {SMOKE_DATASET_PATH}"
assert FULL_DATASET_PATH != SMOKE_DATASET_PATH, "This notebook must use the full ORPO dataset, not the smoke file."

records = [json.loads(line) for line in FULL_DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print({"dataset_path": str(FULL_DATASET_PATH), "records_loaded": len(records)})

required_keys = {"prompt", "chosen", "rejected", "source_partition", "output_partition"}
for index, record in enumerate(records, start=1):
    missing = required_keys - set(record)
    assert not missing, f"Record {index} is missing keys: {missing}"
    assert record["chosen"] != record["rejected"], f"Record {index} has identical chosen and rejected strings"
    assert record["source_partition"] in {"train", "dev"}, f"Unexpected source_partition in record {index}: {record['source_partition']}"
    assert record["output_partition"] != "held_out", f"Held-out leakage found in record {index}"

full_ds = Dataset.from_list([
    {
        "prompt": record["prompt"],
        "chosen": record["chosen"],
        "rejected": record["rejected"],
    }
    for record in records
])

source_partition_counts = {}
for record in records:
    key = record["source_partition"]
    source_partition_counts[key] = source_partition_counts.get(key, 0) + 1

print({
    "dataset_rows": len(full_ds),
    "source_partition_counts": source_partition_counts,
})
print(full_ds[0])


## Model And Explicit LoRA Setup

This uses explicit PEFT wrapping so we can verify the model is a real LoRA / PEFT model before training and save a clean adapter-only artifact at the end.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = "unsloth/Qwen3-0.6B"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = False

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=TARGET_MODULES,
)

model = get_peft_model(model, lora_config)

print("model type after LoRA:", type(model))
print("has peft_config:", hasattr(model, "peft_config"))
print("active_adapter:", getattr(model, "active_adapter", None))
if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()

assert hasattr(model, "peft_config"), "LoRA wrapping failed: model is not PEFT-wrapped."


## ORPO Full Training Run

This cell uses the current TRL ORPO path and includes the fp32 trainable-parameter fix that avoids AMP unscale errors on T4.


In [ ]:
import inspect
import torch
from trl.experimental.orpo import ORPOTrainer, ORPOConfig

OUTPUT_DIR = repo_root / "models" / "orpo_qwen3_0_6b_full_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

orpo_args = ORPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    warmup_steps=5,
    fp16=True,
    bf16=False,
    logging_steps=5,
    save_strategy="no",
    optim="adamw_8bit",
    max_length=MAX_SEQ_LENGTH,
    remove_unused_columns=False,
    report_to=[],
)

trainer_kwargs = {
    "model": model,
    "args": orpo_args,
    "train_dataset": full_ds,
}

trainer_signature = inspect.signature(ORPOTrainer.__init__)
if "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer

trainable_param_count = 0
trainable_dtypes = set()
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()
        trainable_param_count += param.numel()
        trainable_dtypes.add(str(param.dtype))

print({
    "trainable_params": trainable_param_count,
    "trainable_dtypes": sorted(trainable_dtypes),
})

trainer = ORPOTrainer(**trainer_kwargs)
train_result = trainer.train()

loss_values = [
    entry["loss"]
    for entry in trainer.state.log_history
    if isinstance(entry, dict) and "loss" in entry
]

print({
    "global_step": trainer.state.global_step,
    "training_loss": getattr(train_result, "training_loss", None),
    "logged_loss_count": len(loss_values),
    "max_memory_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2),
    "output_dir": str(OUTPUT_DIR),
})


## Save Adapter-Only Artifact And Training Summary

This is the key cell for the final deliverable. It saves the **LoRA adapter only** and verifies that the adapter files exist.


In [ ]:
import datetime as dt
import json
import math
import matplotlib.pyplot as plt
import torch

ADAPTER_DIR = repo_root / "models" / "orpo_qwen3_0_6b_lora_adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

files_in_adapter_dir = sorted(p.name for p in ADAPTER_DIR.iterdir())
print("ADAPTER_DIR:", ADAPTER_DIR)
print("Files:", files_in_adapter_dir)

assert (ADAPTER_DIR / "adapter_config.json").exists(), "Missing adapter_config.json; adapter-only save failed."
assert (ADAPTER_DIR / "adapter_model.safetensors").exists(), "Missing adapter_model.safetensors; adapter-only save failed."

loss_points = [
    entry for entry in trainer.state.log_history
    if isinstance(entry, dict) and "loss" in entry
]
loss_steps = [entry.get("step") for entry in loss_points]
loss_values = [float(entry["loss"]) for entry in loss_points]

assert loss_values, "No loss values were logged during training."
assert all(math.isfinite(loss) for loss in loss_values), f"Non-finite loss detected: {loss_values}"

plt.figure(figsize=(10, 5))
plt.plot(loss_steps, loss_values, marker="o")
plt.title("ORPO Training Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
loss_curve_path = ADAPTER_DIR / "loss_curve.png"
plt.savefig(loss_curve_path, bbox_inches="tight")
plt.show()

training_run_summary = {
    "timestamp_utc": dt.datetime.now(dt.UTC).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
    "status": "pass",
    "dataset_path": str(FULL_DATASET_PATH),
    "record_count": len(records),
    "model_name": MODEL_NAME,
    "precision": "fp16",
    "qlora": LOAD_IN_4BIT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "orpo_beta": 0.1,
    "learning_rate": 2e-5,
    "batch_size": 1,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 1,
    "global_step": trainer.state.global_step,
    "loss_values": loss_values,
    "all_losses_finite": True,
    "adapter_dir": str(ADAPTER_DIR),
    "adapter_config_path": str(ADAPTER_DIR / "adapter_config.json"),
    "adapter_model_path": str(ADAPTER_DIR / "adapter_model.safetensors"),
    "loss_curve_path": str(loss_curve_path),
    "max_memory_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2),
}

summary_path = ADAPTER_DIR / "training_run_summary.json"
summary_path.write_text(json.dumps(training_run_summary, indent=2), encoding="utf-8")

print(json.dumps(training_run_summary, indent=2))


## Optional Zip Download Cell

Use this only after the adapter-only save succeeds and you want a single archive to download back from Colab.


In [ ]:
!cd "/content/Sales Eval Bench/models" && zip -r orpo_qwen3_0_6b_lora_adapter.zip orpo_qwen3_0_6b_lora_adapter


## After The Run

Copy these files back into your repo from `models/orpo_qwen3_0_6b_lora_adapter`:

- `adapter_config.json`
- `adapter_model.safetensors`
- `training_run_summary.json`
- `loss_curve.png`
- tokenizer files

Then update `docs/progress.md` and move on to the Act IV held-out ablations.
